# Baseline: исходная VLM на независимых тестах VK

Здесь измеряется качество модели **до дообучения**. Это точка отсчёта, с которой затем будут сравниваться все LoRA-варианты. Для оценки используются только `deepvk/GQA-ru` и `deepvk/MMBench-ru`; `LLaVA-Instruct-ru` не участвует в этом ноутбуке.

Основная модель эксперимента — `Qwen/Qwen2.5-VL-3B-Instruct` в 4-битном режиме. На RTX 4060 с 8 ГБ памяти это реалистичный вариант для inference и QLoRA.

## План версий модели

| Версия | Подход | Назначение |
| --- | --- | --- |
| B0 | Исходная Qwen2.5-VL-3B-Instruct, без обучения | Baseline |
| F1 | QLoRA, 12 000 примеров, LoRA rank 8 | Первая проверка пайплайна |
| F2 | QLoRA, 12 000 примеров, LoRA rank 16 | Проверка влияния ёмкости адаптера |
| F3 | Лучшие параметры + увеличенная выборка | Итоговая версия |

Все версии оцениваются на одних и тех же фиксированных 100 примерах каждого бенчмарка. Полная оценка будет выполнена для лучшего варианта после проверки пайплайна.

In [1]:
# При первом запуске в новом окружении: 
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
# %pip install transformers datasets accelerate peft bitsandbytes qwen-vl-utils sentencepiece

import os
from pathlib import Path
import re
import time

import pandas as pd
from IPython.display import Markdown, display
# В этой сети Xet-загрузчик Hugging Face не устанавливает соединение;
# обычная HTTP-загрузка репозитория работает стабильно.
os.environ['HF_HUB_DISABLE_XET'] = '1'

import torch
from datasets import load_dataset
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
SEED = 42
GQA_SIZE = 100
MMBENCH_SIZE = 100
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

assert torch.cuda.is_available(), 'Для baseline нужна CUDA; проверьте установку PyTorch.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

GPU: NVIDIA GeForce RTX 4060
CUDA: 12.6


## 1. Фиксированные тестовые подвыборки

GQA-ru содержит таблицы вопросов и изображений отдельно, поэтому объединяем их по `imageId`. У GQA эталонный ответ — одно слово; для MMBench модель обязана вернуть только латинскую букву варианта.

In [2]:
gqa_questions = load_dataset('deepvk/GQA-ru', 'testdev_balanced_instructions', split='testdev')
gqa_images = load_dataset('deepvk/GQA-ru', 'testdev_balanced_images', split='testdev')
gqa_image_by_id = {row['id']: row['image'] for row in gqa_images}

gqa_df = pd.DataFrame(gqa_questions).rename(columns={'imageId': 'image_id'})
gqa_df['image'] = gqa_df['image_id'].map(gqa_image_by_id)
assert gqa_df['image'].notna().all(), 'Не для всех вопросов GQA нашлось изображение.'
gqa_eval = gqa_df.sample(GQA_SIZE, random_state=SEED).reset_index(drop=True)

mmbench = load_dataset('deepvk/MMBench-ru', split='dev')
mmbench_df = pd.DataFrame(mmbench)
mmbench_eval = mmbench_df.sample(MMBENCH_SIZE, random_state=SEED).reset_index(drop=True)

print(f'GQA-ru: выбрано {len(gqa_eval)} из {len(gqa_df)} заданий; изображений в полном тесте: {len(gqa_images)}')
print(f'MMBench-ru: выбрано {len(mmbench_eval)} из {len(mmbench_df)} заданий')
display(gqa_eval[['question', 'answer']].head(3))
display(mmbench_eval[['question', 'A', 'B', 'C', 'D', 'answer']].head(2))

GQA-ru: выбрано 100 из 12216 заданий; изображений в полном тесте: 398
MMBench-ru: выбрано 100 из 3910 заданий


,question,answer
0,Человек сверху или снизу?,сверху
1,"Вы видите и овец, и коз?",да
2,Какого цвета фургон?,черный


,question,A,B,C,D,answer
0,В каком углу нет тарелок?,внизу-слева,внизу-справа,вверху-справа,вверху-слева,A
1,Какое настроение передает это изображение?,Счастье,Злость,Грусть,Тревога,B


## 2. Загрузка исходной модели

4-битная загрузка сокращает потребление видеопамяти. Параметры исходной модели не изменяются: это именно zero-shot baseline.

In [3]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map='auto',
    torch_dtype=torch.float16,
).eval()
print('Модель загружена. Использование GPU, МБ:', round(torch.cuda.memory_allocated() / 1024**2))

W0714 20:27:14.912000 14256 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Модель загружена. Использование GPU, МБ: 2464


In [4]:
def generate_answer(image, prompt, max_new_tokens=32):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = generated[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

def normalize_gqa(text):
    return re.sub(r'[^а-яёa-z0-9-]', '', text.lower())

def extract_option(text):
    match = re.search(r'(?<![A-ZА-Я])[ABCD](?![A-ZА-Я])', text.upper())
    return match.group(0) if match else ''

## 3. Оценка baseline

Генерация выполняется с `do_sample=False`, поэтому при одинаковой версии модели и данных результат воспроизводим. Для GQA используется Exact Match нормализованного ответа; для MMBench — точное совпадение буквы варианта.

In [5]:
gqa_rows = []
started = time.perf_counter()
for i, row in gqa_eval.iterrows():
    prompt = f"Вопрос: {row.question}\nОтветь одним словом на русском языке, без пояснений."
    prediction = generate_answer(row.image, prompt, max_new_tokens=12)
    gqa_rows.append({
        'benchmark': 'GQA-ru', 'sample_id': row.id, 'question': row.question,
        'target': row.answer, 'prediction': prediction,
        'is_correct': normalize_gqa(prediction) == normalize_gqa(row.answer),
    })
    if (i + 1) % 10 == 0:
        print(f'GQA-ru: {i + 1}/{len(gqa_eval)}')
gqa_results = pd.DataFrame(gqa_rows)
print(f'GQA-ru Exact Match: {gqa_results.is_correct.mean():.1%}; время: {(time.perf_counter() - started) / 60:.1f} мин.')

c:\Users\miste\miniconda3\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


GQA-ru: 10/100


GQA-ru: 20/100


GQA-ru: 30/100


GQA-ru: 40/100


...

GQA-ru: 60/100


GQA-ru: 70/100


GQA-ru: 80/100


GQA-ru: 90/100


GQA-ru: 100/100
GQA-ru Exact Match: 43.0%; время: 0.6 мин.


In [6]:
mmbench_rows = []
started = time.perf_counter()
for i, row in mmbench_eval.iterrows():
    options = '\n'.join(f'{letter}. {row[letter]}' for letter in 'ABCD')
    prompt = (f"Вопрос: {row.question}\n{options}\n\n"
              'Выбери правильный вариант. Ответь только одной латинской буквой: A, B, C или D.')
    prediction = generate_answer(row.image, prompt, max_new_tokens=8)
    option = extract_option(prediction)
    mmbench_rows.append({
        'benchmark': 'MMBench-ru', 'sample_id': row['index'], 'question': row.question,
        'target': row.answer, 'prediction': prediction, 'selected_option': option,
        'is_correct': option == row.answer,
    })
    if (i + 1) % 10 == 0:
        print(f'MMBench-ru: {i + 1}/{len(mmbench_eval)}')
mmbench_results = pd.DataFrame(mmbench_rows)
print(f'MMBench-ru Accuracy: {mmbench_results.is_correct.mean():.1%}; время: {(time.perf_counter() - started) / 60:.1f} мин.')

MMBench-ru: 10/100


MMBench-ru: 20/100


MMBench-ru: 30/100


MMBench-ru: 40/100


MMBench-ru: 50/100


MMBench-ru: 60/100


MMBench-ru: 70/100


MMBench-ru: 80/100


MMBench-ru: 90/100


MMBench-ru: 100/100
MMBench-ru Accuracy: 74.0%; время: 0.4 мин.


In [7]:
baseline_results = pd.concat([gqa_results, mmbench_results], ignore_index=True)
baseline_results.to_csv(RESULTS_DIR / 'baseline_predictions.csv', index=False)
metrics = (baseline_results.groupby('benchmark')['is_correct'].agg(['mean', 'sum', 'count'])
           .rename(columns={'mean': 'metric', 'sum': 'correct', 'count': 'examples'}))
metrics['metric'] = (metrics['metric'] * 100).round(1)
metrics.to_csv(RESULTS_DIR / 'baseline_metrics.csv')
display(metrics)

display(Markdown(
    '### Вывод baseline\n'
    f"- На фиксированных выборках baseline получен без обучения; эти значения будут точкой сравнения для F1–F3.\n"
    f"- GQA-ru Exact Match: **{metrics.loc['GQA-ru', 'metric']:.1f}%**.\n"
    f"- MMBench-ru Accuracy: **{metrics.loc['MMBench-ru', 'metric']:.1f}%**.\n"
    '- Файл с каждым предсказанием сохранён в `results/baseline_predictions.csv`; он нужен для качественного анализа ошибок.'
))

display(baseline_results.query('not is_correct')[['benchmark', 'question', 'target', 'prediction']].head(10))

,metric,correct,examples
benchmark,,,
GQA-ru,43.0,43,100
MMBench-ru,74.0,74,100


### Вывод baseline
- На фиксированных выборках baseline получен без обучения; эти значения будут точкой сравнения для F1–F3.
- GQA-ru Exact Match: **43.0%**.
- MMBench-ru Accuracy: **74.0%**.
- Файл с каждым предсказанием сохранён в `results/baseline_predictions.csv`; он нужен для качественного анализа ошибок.

,benchmark,question,target,prediction
0,GQA-ru,Человек сверху или снизу?,сверху,Снизу
1,GQA-ru,"Вы видите и овец, и коз?",да,Нет
2,GQA-ru,Какого цвета фургон?,черный,оранжевый
3,GQA-ru,Какой длины стол?,длинный,короткий
5,GQA-ru,"Как называется предмет мебели, стоящий на ковре?",стол,Кресло
7,GQA-ru,Какая еда не жёлтая?,пепперони,Пицца
8,GQA-ru,Какой вид транспортного средства припаркован?,автомобили,Автомобиль
11,GQA-ru,Что находится на полу?,диван,Ковер
13,GQA-ru,Какое устройство находится слева от стула?,роутер,Компьютер
14,GQA-ru,Кто носит обувь?,футболист,Девушки
